# Dataset Inspector
**Utility for checking data types in dataset folders**

In [1]:
import os
import sys
from pathlib import Path
import json

# List of dataset folders to inspect
DATASET_FOLDERS = [
    r'C:\Users\Alfred\Desktop\cbsd-2',
    
]

def inspect_folder(folder_path):
    """Inspect a folder and return its contents and file types."""
    if not os.path.exists(folder_path):
        return {"error": "Folder does not exist", "path": folder_path}
    
    result = {
        "path": folder_path,
        "exists": True,
        "is_directory": os.path.isdir(folder_path),
        "contents": [],
        "file_types": {},
        "total_files": 0,
        "total_folders": 0
    }
    
    try:
        for item in os.listdir(folder_path):
            item_path = os.path.join(folder_path, item)
            item_info = {
                "name": item,
                "is_file": os.path.isfile(item_path),
                "is_directory": os.path.isdir(item_path),
                "size_bytes": os.path.getsize(item_path) if os.path.isfile(item_path) else None
            }
            
            # Get file extension
            if os.path.isfile(item_path):
                ext = Path(item).suffix.lower()
                if ext:
                    result["file_types"][ext] = result["file_types"].get(ext, 0) + 1
            
            result["contents"].append(item_info)
            
            if os.path.isfile(item_path):
                result["total_files"] += 1
            elif os.path.isdir(item_path):
                result["total_folders"] += 1
                
    except PermissionError:
        result["error"] = "Permission denied"
    except Exception as e:
        result["error"] = str(e)
    
    return result

def inspect_image_folder(folder_path):
    """Inspect an image folder and check for valid image files."""
    valid_extensions = {'.jpg', '.jpeg', '.png', '.gif', '.bmp', '.webp', '.tiff', '.tif'}
    
    result = {
        "path": folder_path,
        "image_files": [],
        "non_image_files": [],
        "subfolders": [],
        "total_images": 0,
        "image_types": {}
    }
    
    if not os.path.exists(folder_path):
        result["error"] = "Folder does not exist"
        return result
    
    for item in os.listdir(folder_path):
        item_path = os.path.join(folder_path, item)
        
        if os.path.isdir(item_path):
            result["subfolders"].append(item)
        elif os.path.isfile(item_path):
            ext = Path(item).suffix.lower()
            if ext in valid_extensions:
                result["image_files"].append({
                    "filename": item,
                    "extension": ext,
                    "size_bytes": os.path.getsize(item_path)
                })
                result["image_types"][ext] = result["image_types"].get(ext, 0) + 1
                result["total_images"] += 1
            else:
                result["non_image_files"].append({
                    "filename": item,
                    "extension": ext,
                    "size_bytes": os.path.getsize(item_path)
                })
    
    return result

def print_inspection(result):
    """Pretty print the inspection results."""
    print("=" * 60)
    print(f"Folder: {result.get('path', 'N/A')}")
    print("=" * 60)
    
    if "error" in result:
        print(f"❌ Error: {result['error']}")
        return
    
    print(f"✅ Exists: {result.get('exists', False)}")
    print(f"📁 Total Files: {result.get('total_files', 0)}")
    print(f"📂 Total Subfolders: {result.get('total_folders', 0)}")
    
    if result.get('file_types'):
        print("\n📊 File Types Distribution:")
        for ext, count in sorted(result['file_types'].items()):
            print(f"   {ext}: {count} files")
    
    if result.get('contents'):
        print("\n📋 Contents:")
        for item in result['contents'][:10]:  # Show first 10
            item_type = "📄" if item['is_file'] else "📁"
            size_str = f" ({item['size_bytes']:,} bytes)" if item['size_bytes'] else ""
            print(f"   {item_type} {item['name']}{size_str}")
        if len(result['contents']) > 10:
            print(f"   ... and {len(result['contents']) - 10} more items")

## Inspect Dataset Folders

In [5]:
# Inspect all dataset folders
for folder in DATASET_FOLDERS:
    result = inspect_folder(folder)
    print_inspection(result)
    print()

Folder: C:\Users\Alfred\Desktop\cbsd-2
✅ Exists: True
📁 Total Files: 2420
📂 Total Subfolders: 0

📊 File Types Distribution:
   .jpg: 1210 files
   .xml: 1210 files

📋 Contents:
   📄 1617105843672.jpg (836,357 bytes)
   📄 1617105843672.xml (683 bytes)
   📄 1617105851020.jpg (940,583 bytes)
   📄 1617105851020.xml (682 bytes)
   📄 1617105852257.jpg (1,137,655 bytes)
   📄 1617105852257.xml (684 bytes)
   📄 1617105856605.jpg (784,609 bytes)
   📄 1617105856605.xml (683 bytes)
   📄 1617105859917.jpg (478,162 bytes)
   📄 1617105859917.xml (683 bytes)
   ... and 2410 more items



## Inspect Image Data (if data folder exists)

In [6]:
# Check the main data folder for image classes
data_folder = r'C:\Users\Alfred\Desktop\cbsd-2'

if os.path.exists(data_folder):
    print("=" * 60)
    print(f"Inspecting image data in: {data_folder}")
    print("=" * 60)
    
    # List subfolders (classes)
    classes = [d for d in os.listdir(data_folder) if os.path.isdir(os.path.join(data_folder, d))]
    print(f"\n📂 Found {len(classes)} class folders:")
    
    for cls in sorted(classes):
        cls_path = os.path.join(data_folder, cls)
        images = [f for f in os.listdir(cls_path) if f.lower().endswith(('.jpg', '.jpeg', '.png', '.gif', '.bmp', '.webp', '.tiff', '.tif'))]
        print(f"   📁 {cls}: {len(images)} images")
else:
    print(f"⚠️  Data folder not found: {data_folder}")

Inspecting image data in: C:\Users\Alfred\Desktop\cbsd-2

📂 Found 0 class folders:


## Custom Folder Inspection

In [ ]:
# Inspect a custom folder (uncomment and modify as needed)
# custom_folder = r'C:\Users\Alfred\Downloads\your_dataset_here'
# result = inspect_folder(custom_folder)
# print_inspection(result)